In [ ]:
"""
Client-Focused Product Recommendation Engine for Financial Advisers
Author: AI Data Scientist Assistant
Date: 2025
Description: Recommendation engine based purely on CLIENT demographics, not adviser experience
"""

import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    precision_recall_curve, roc_auc_score, average_precision_score
)
import lightgbm as lgb
import joblib
from typing import Dict, List
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
np.random.seed(42)

# ============================================================================
# PART 1: CLIENT-FOCUSED FEATURE ENGINEERING
# ============================================================================

class ClientFeatureEngineer:
    """Feature engineering based purely on CLIENT characteristics"""
    
    def __init__(self):
        self.fitted = False
        
    def fit(self, df: pd.DataFrame) -> 'ClientFeatureEngineer':
        """Fit the feature engineer (minimal fitting needed for client features)"""
        self.fitted = True
        return self
    
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Generate client-based features only"""
        if not self.fitted:
            raise ValueError("ClientFeatureEngineer must be fitted before transform")
        
        df = df.copy()
        
        # Create all client-focused features
        df = self._create_demographic_features(df)
        df = self._create_life_stage_features(df)
        df = self._create_financial_capacity_features(df)
        df = self._create_insurance_need_features(df)
        
        return df
    
    def _create_demographic_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Create basic demographic features and interactions"""
        
        # Convert to ordinal for tree models
        df['Age_Ordinal'] = df['AgeGroup'].map({'20s': 1, '30s': 2, '40s': 3, '50s': 4})
        df['Income_Ordinal'] = df['IncomeBracket'].map({'<50k': 1, '>50k': 2, '>150k': 3})
        
        # Binary flags for key segments
        df['Is_Male'] = (df['Gender'] == 'Male').astype(int)
        df['Is_Female'] = (df['Gender'] == 'Female').astype(int)
        df['Is_Single'] = (df['MaritalStatus'] == 'Single').astype(int)
        df['Is_Married'] = df['MaritalStatus'].str.contains('Married').astype(int)
        df['Has_Dependents'] = df['MaritalStatus'].str.contains('dependents').astype(int)
        
        # Age groups binary
        df['Is_Young'] = df['AgeGroup'].isin(['20s', '30s']).astype(int)
        df['Is_Middle_Aged'] = df['AgeGroup'].isin(['40s']).astype(int)
        df['Is_Senior'] = df['AgeGroup'].isin(['50s']).astype(int)
        
        # Income segments
        df['Is_LowIncome'] = (df['IncomeBracket'] == '<50k').astype(int)
        df['Is_MiddleIncome'] = (df['IncomeBracket'] == '>50k').astype(int)
        df['Is_HighIncome'] = (df['IncomeBracket'] == '>150k').astype(int)
        
        return df
    
    def _create_life_stage_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Create life stage features based on age and family status"""
        
        # Key life stages
        df['Young_Single'] = ((df['Is_Young'] == 1) & (df['Is_Single'] == 1)).astype(int)
        df['Young_Family'] = ((df['Is_Young'] == 1) & (df['Has_Dependents'] == 1)).astype(int)
        df['Young_Married_NoDeps'] = ((df['Is_Young'] == 1) & (df['Is_Married'] == 1) & (df['Has_Dependents'] == 0)).astype(int)
        
        df['MidAge_Single'] = ((df['Is_Middle_Aged'] == 1) & (df['Is_Single'] == 1)).astype(int)
        df['MidAge_Family'] = ((df['Is_Middle_Aged'] == 1) & (df['Has_Dependents'] == 1)).astype(int)
        
        df['Senior_Empty_Nest'] = ((df['Is_Senior'] == 1) & (df['Has_Dependents'] == 0)).astype(int)
        df['Senior_With_Deps'] = ((df['Is_Senior'] == 1) & (df['Has_Dependents'] == 1)).astype(int)
        
        # Career/Life progression score
        df['Life_Progress_Score'] = df['Age_Ordinal'] + df['Income_Ordinal'] - df['Has_Dependents']
        
        return df
    
    def _create_financial_capacity_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Create features indicating financial capacity and investment potential"""
        
        # Wealth indicators
        df['Wealth_Score'] = df['Age_Ordinal'] * df['Income_Ordinal']
        df['High_Wealth'] = (df['Wealth_Score'] >= 9).astype(int)  # 40s/50s with high income
        df['Low_Wealth'] = (df['Wealth_Score'] <= 4).astype(int)   # Young or low income
        
        # Investment capacity (income minus family obligations)
        df['Investment_Capacity'] = df['Income_Ordinal'] * 2 - df['Has_Dependents'] * 2
        
        # Special segments
        df['Young_HighEarner'] = ((df['Is_Young'] == 1) & (df['Is_HighIncome'] == 1)).astype(int)
        df['Senior_HighEarner'] = ((df['Is_Senior'] == 1) & (df['Is_HighIncome'] == 1)).astype(int)
        df['Single_HighEarner'] = ((df['Is_Single'] == 1) & (df['Is_HighIncome'] == 1)).astype(int)
        
        # Financial stress indicator
        df['Financial_Stress'] = ((df['Is_LowIncome'] == 1) & (df['Has_Dependents'] == 1)).astype(int)
        
        return df
    
    def _create_insurance_need_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Create features indicating insurance and protection needs"""
        
        # Protection need score
        df['Protection_Need'] = (
            df['Has_Dependents'] * 3 +  # Highest weight for dependents
            df['Is_Married'] * 2 +       # Medium weight for married
            df['Is_Young'] * 1            # Some weight for young (future planning)
        )
        
        # Life insurance indicators
        df['High_Life_Insurance_Need'] = ((df['Has_Dependents'] == 1) & (df['Age_Ordinal'] <= 3)).astype(int)
        df['Low_Life_Insurance_Need'] = ((df['Has_Dependents'] == 0) & (df['Is_Single'] == 1)).astype(int)
        
        # Retirement planning indicators
        df['Near_Retirement'] = (df['Is_Senior'] == 1).astype(int)
        df['Mid_Career'] = (df['Age_Ordinal'] == 2).astype(int)
        
        # Risk profile (younger and higher income = higher risk tolerance)
        df['Risk_Tolerance'] = (5 - df['Age_Ordinal']) + df['Income_Ordinal']
        
        return df

# ============================================================================
# PART 2: SIMPLIFIED FEATURE PROCESSOR
# ============================================================================

class ClientFeatureProcessor:
    """Process client features for modeling"""
    
    def __init__(self, feature_engineer: ClientFeatureEngineer):
        self.feature_engineer = feature_engineer
        self.feature_columns = None
        
    def prepare_features(self, df: pd.DataFrame, is_training: bool = True) -> pd.DataFrame:
        """Prepare feature matrix focusing on client characteristics"""
        
        # Apply feature engineering
        df_eng = self.feature_engineer.transform(df)
        
        # Select features (all numeric now)
        feature_cols = [
            # Ordinal features
            'Age_Ordinal', 'Income_Ordinal',
            
            # Demographics
            'Is_Male', 'Is_Female', 'Is_Single', 'Is_Married', 'Has_Dependents',
            'Is_Young', 'Is_Middle_Aged', 'Is_Senior',
            'Is_LowIncome', 'Is_MiddleIncome', 'Is_HighIncome',
            
            # Life stages
            'Young_Single', 'Young_Family', 'Young_Married_NoDeps',
            'MidAge_Single', 'MidAge_Family',
            'Senior_Empty_Nest', 'Senior_With_Deps',
            
            # Financial capacity
            'Wealth_Score', 'High_Wealth', 'Low_Wealth',
            'Investment_Capacity', 'Life_Progress_Score',
            'Young_HighEarner', 'Senior_HighEarner', 'Single_HighEarner',
            'Financial_Stress',
            
            # Insurance needs
            'Protection_Need', 'High_Life_Insurance_Need', 'Low_Life_Insurance_Need',
            'Near_Retirement', 'Mid_Career', 'Risk_Tolerance'
        ]
        
        df_final = df_eng[feature_cols].copy()
        
        if is_training:
            self.feature_columns = df_final.columns.tolist()
        else:
            # Ensure same features as training
            for col in self.feature_columns:
                if col not in df_final.columns:
                    df_final[col] = 0
            df_final = df_final[self.feature_columns]
        
        return df_final

# ============================================================================
# PART 3: SIMPLIFIED MODEL (Same as before but cleaner)
# ============================================================================

class ClientBasedRecommendationModel:
    """Recommendation model based purely on client characteristics"""
    
    def __init__(self, n_jobs: int = -1):
        self.models = {}
        self.thresholds = {}
        self.product_columns = None
        self.n_jobs = n_jobs
        
    def create_base_model(self) -> lgb.LGBMClassifier:
        """Create optimized LightGBM model for client-based features"""
        return lgb.LGBMClassifier(
            n_estimators=100,      # Fewer features = fewer trees needed
            max_depth=5,           # Shallower trees for simpler features
            num_leaves=20,
            min_child_samples=50,  # More conservative to prevent overfitting
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.7,
            reg_alpha=0.2,         # More regularization
            reg_lambda=0.3,
            class_weight='balanced',
            random_state=42,
            n_jobs=1,
            verbose=-1
        )
    
    def fit(self, X: pd.DataFrame, y: pd.DataFrame, y_weights: pd.DataFrame = None,
            val_size: float = 0.2):
        """Train models for all products"""
        
        self.product_columns = y.columns.tolist()
        
        # Create stratification column - limit to avoid rare class issues
        # Use sum of products but cap it to avoid too many unique values
        stratify_col = y.sum(axis=1).clip(upper=5)
        
        # Check if stratification is possible
        min_class_count = stratify_col.value_counts().min()
        
        if min_class_count < 2:
            # If stratification isn't possible, do random split
            print("Warning: Using random split due to class imbalance")
            if y_weights is not None:
                X_train, X_val, y_train, y_val, weights_train, weights_val = train_test_split(
                    X, y, y_weights, test_size=val_size, random_state=42
                )
            else:
                X_train, X_val, y_train, y_val = train_test_split(
                    X, y, test_size=val_size, random_state=42
                )
                weights_train = pd.DataFrame(1, index=y_train.index, columns=y_train.columns)
        else:
            # Use stratified split
            if y_weights is not None:
                X_train, X_val, y_train, y_val, weights_train, weights_val = train_test_split(
                    X, y, y_weights, test_size=val_size, random_state=42, 
                    stratify=stratify_col
                )
            else:
                X_train, X_val, y_train, y_val = train_test_split(
                    X, y, test_size=val_size, random_state=42, 
                    stratify=stratify_col
                )
                weights_train = pd.DataFrame(1, index=y_train.index, columns=y_train.columns)
        
        print(f"Training on {len(X_train)} samples, validating on {len(X_val)} samples")
        print(f"Training {len(self.product_columns)} product models...")
        
        # Train a model for each product
        for i, product in enumerate(self.product_columns):
            if (i + 1) % 10 == 0:
                print(f"  Training model {i + 1}/{len(self.product_columns)}")
            
            y_product_train = y_train[product]
            y_product_val = y_val[product]
            
            # Skip if no positive samples
            if y_product_train.sum() == 0:
                print(f"  Warning: No positive samples for {product}, skipping...")
                self.models[product] = None
                continue
            
            # Create sample weights if provided
            if y_weights is not None:
                sample_weights = weights_train[product].map({0: 1, 1: 2, 2: 4}).fillna(1)
            else:
                sample_weights = None
            
            # Train model
            model = self.create_base_model()
            
            if sample_weights is not None:
                model.fit(X_train, y_product_train, sample_weight=sample_weights)
            else:
                model.fit(X_train, y_product_train)
            
            self.models[product] = model
            
            # Optimize threshold
            y_scores = model.predict_proba(X_val)[:, 1]
            self.thresholds[product] = self._optimize_threshold(
                y_product_val, y_scores, target_recall=0.7
            )
        
        print("Training complete!")
        return self
    
    def _optimize_threshold(self, y_true: np.ndarray, y_scores: np.ndarray, 
                           target_recall: float = 0.7) -> float:
        """Find threshold that achieves target recall"""
        
        precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
        
        # Find threshold closest to target recall
        recall_diff = np.abs(recall[:-1] - target_recall)
        best_idx = np.argmin(recall_diff)
        
        # If we can't achieve target recall, use the threshold that maximizes F1
        if recall[best_idx] < target_recall * 0.8:
            f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-10)
            best_idx = np.argmax(f1_scores)
        
        return thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    
    def predict_proba(self, X: pd.DataFrame) -> pd.DataFrame:
        """Get probability predictions for all products"""
        
        predictions = pd.DataFrame(index=X.index, columns=self.product_columns)
        
        for product in self.product_columns:
            if product in self.models and self.models[product] is not None:
                predictions[product] = self.models[product].predict_proba(X)[:, 1]
            else:
                predictions[product] = 0.0
        
        return predictions
    
    def recommend_top_k(self, X: pd.DataFrame, k: int = 3) -> List[Dict]:
        """Get top-k product recommendations for each client"""
        
        probas = self.predict_proba(X)
        recommendations = []
        
        for idx in range(len(X)):
            client_probas = probas.iloc[idx]
            top_products = client_probas.nlargest(k)
            
            client_recs = {
                'client_index': idx,
                'recommendations': [
                    {
                        'rank': i + 1,
                        'product': product,
                        'probability': float(prob),
                        'confidence': 'High' if prob >= 0.7 else 'Medium' if prob >= 0.4 else 'Low'
                    }
                    for i, (product, prob) in enumerate(top_products.items())
                ]
            }
            recommendations.append(client_recs)
        
        return recommendations

# ============================================================================
# PART 4: SIMPLIFIED MAIN PIPELINE
# ============================================================================

def main():
    """Main execution pipeline - CLIENT FOCUSED"""
    
    print("=" * 80)
    print("CLIENT-FOCUSED PRODUCT RECOMMENDATION ENGINE")
    print("(No Adviser Experience Features)")
    print("=" * 80)
    
    # -------------------------------------------------------------------------
    # Step 1: Data Loading
    # -------------------------------------------------------------------------
    print("\n📁 Step 1: Data Loading")
    print("-" * 40)
    
    df = pd.read_csv('adviser_survey.csv')
    print(f"Loaded {len(df)} samples")
    
    # Filter to clients with at least one rating
    client_features = ['AgeGroup', 'Gender', 'MaritalStatus', 'IncomeBracket']
    rating_cols = [col for col in df.columns if col not in client_features + ['ClientID', 'YearsExperience']]
    
    # Clean ratings
    for col in rating_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    
    # Filter clients with any interest
    mask_any_rating = df[rating_cols].isin([1, 2]).any(axis=1)
    df = df[mask_any_rating].reset_index(drop=True)
    print(f"Filtered to {len(df)} clients with at least one rating")
    
    # -------------------------------------------------------------------------
    # Step 2: Prepare Features and Targets
    # -------------------------------------------------------------------------
    print("\n🎯 Step 2: Prepare Features and Targets")
    print("-" * 40)
    
    # Group rare products if they exist
    rare_products = ['Refundable_Term', 'Other_H&S', 'Other_A&H', 'RP_WL_Protection_with_multiplier']
    existing_rare = [col for col in rare_products if col in df.columns]
    
    if existing_rare:
        df['Other_Specialty'] = df[existing_rare].max(axis=1)
        product_cols = [col for col in rating_cols if col not in existing_rare]
    else:
        product_cols = rating_cols
    
    # Prepare CLIENT features only (no YearsExperience)
    X_raw = df[client_features]
    
    # Binary targets and weights
    y_binary = (df[product_cols] >= 1).astype(int)
    y_weights = df[product_cols].copy()
    
    print(f"Client features: {client_features}")
    print(f"Number of products: {len(product_cols)}")
    print(f"Average products per client (any interest): {y_binary.sum(axis=1).mean():.2f}")
    print(f"Overall positive rate: {y_binary.values.mean():.2%}")
    
    # -------------------------------------------------------------------------
    # Step 3: Feature Engineering
    # -------------------------------------------------------------------------
    print("\n🔧 Step 3: Client Feature Engineering")
    print("-" * 40)
    
    # Initialize and fit feature engineer
    feature_engineer = ClientFeatureEngineer()
    feature_engineer.fit(X_raw)
    
    # Process features
    processor = ClientFeatureProcessor(feature_engineer)
    X_processed = processor.prepare_features(X_raw, is_training=True)
    
    print(f"Engineered features: {X_processed.shape[1]} features")
    print(f"Sample features: {list(X_processed.columns[:10])}")
    
    # -------------------------------------------------------------------------
    # Step 4: Train-Test Split
    # -------------------------------------------------------------------------
    print("\n✂️ Step 4: Train-Test Split")
    print("-" * 40)
    
    # Try stratified split, fall back to random if needed
    try:
        stratify_col = y_binary.sum(axis=1).clip(upper=5)
        X_train, X_test, y_train, y_test, y_weights_train, y_weights_test = train_test_split(
            X_processed, y_binary, y_weights, test_size=0.2, random_state=42,
            stratify=stratify_col
        )
        print("Using stratified split")
    except ValueError as e:
        print(f"Stratification failed: {e}")
        print("Using random split instead")
        X_train, X_test, y_train, y_test, y_weights_train, y_weights_test = train_test_split(
            X_processed, y_binary, y_weights, test_size=0.2, random_state=42
        )
    
    print(f"Training set: {len(X_train)} samples")
    print(f"Test set: {len(X_test)} samples")
    
    # -------------------------------------------------------------------------
    # Step 5: Model Training
    # -------------------------------------------------------------------------
    print("\n🎓 Step 5: Model Training")
    print("-" * 40)
    
    model = ClientBasedRecommendationModel(n_jobs=-1)
    model.fit(X_train, y_train, y_weights=y_weights_train, val_size=0.2)
    
    print(f"Trained {len([m for m in model.models.values() if m is not None])} product models")
    
    # -------------------------------------------------------------------------
    # Step 6: Evaluation
    # -------------------------------------------------------------------------
    print("\n📊 Step 6: Model Evaluation")
    print("-" * 40)
    
    # Get predictions
    y_pred_proba = model.predict_proba(X_test)
    recommendations = model.recommend_top_k(X_test, k=3)
    
    # Calculate metrics
    from sklearn.metrics import classification_report
    
    # Overall metrics
    y_true_flat = y_test.values.flatten()
    y_pred_flat = (y_pred_proba.values >= 0.5).flatten()
    y_proba_flat = y_pred_proba.values.flatten()
    
    precision = np.sum((y_pred_flat == 1) & (y_true_flat == 1)) / (np.sum(y_pred_flat == 1) + 1e-10)
    recall = np.sum((y_pred_flat == 1) & (y_true_flat == 1)) / (np.sum(y_true_flat == 1) + 1e-10)
    f1 = 2 * precision * recall / (precision + recall + 1e-10)
    
    print(f"Overall Precision: {precision:.3f}")
    print(f"Overall Recall: {recall:.3f}")
    print(f"Overall F1: {f1:.3f}")
    
    # Coverage metrics
    coverage_count = 0
    high_interest_coverage = 0
    
    for i, rec in enumerate(recommendations):
        true_products = y_test.iloc[i]
        true_positives = set(true_products[true_products == 1].index)
        recommended = [r['product'] for r in rec['recommendations']]
        
        if len(set(recommended) & true_positives) > 0:
            coverage_count += 1
        
        high_interest = set(y_weights_test.iloc[i][y_weights_test.iloc[i] == 2].index)
        if len(set(recommended) & high_interest) > 0:
            high_interest_coverage += 1
    
    print(f"\nCoverage@3: {coverage_count / len(recommendations):.2%}")
    print(f"High Interest Coverage@3: {high_interest_coverage / len(recommendations):.2%}")
    
    # -------------------------------------------------------------------------
    # Step 7: Feature Importance
    # -------------------------------------------------------------------------
    print("\n🔍 Step 7: Feature Importance Analysis")
    print("-" * 40)
    
    # Get average feature importance across all models
    importance_dict = {}
    for product, product_model in model.models.items():  # Changed variable name to avoid confusion
        if product_model is not None and hasattr(product_model, 'feature_importances_'):
            importance_dict[product] = product_model.feature_importances_
    
    if importance_dict:
        importance_df = pd.DataFrame(importance_dict, index=X_train.columns)
        importance_df['avg_importance'] = importance_df.mean(axis=1)
        importance_df = importance_df.sort_values('avg_importance', ascending=False)
        
        print("\nTop 10 Most Important CLIENT Features:")
        for i, (feat, imp) in enumerate(importance_df['avg_importance'].head(10).items()):
            print(f"  {i+1:2d}. {feat:30s}: {imp:.4f}")
    
    # -------------------------------------------------------------------------
    # Step 8: Sample Recommendations
    # -------------------------------------------------------------------------
    print("\n💡 Step 8: Sample Recommendations")
    print("-" * 40)
    
    sample_recs = model.recommend_top_k(X_test.iloc[:3], k=3)
    
    for i, rec in enumerate(sample_recs):

        
        print(f"\n👤 Test Client {i+1}:")
        client_row = X_test.iloc[i]
        print(f"  Key features:")
        print(f"    Age: {client_row['Age_Ordinal']}, Income: {client_row['Income_Ordinal']}")
        print(f"    Has Dependents: {client_row['Has_Dependents']}")
        print(f"  Recommendations:")
        for r in rec['recommendations']:
            print(f"    {r['rank']}. {r['product']:25s} (prob={r['probability']:.2%}, conf={r['confidence']})")
    
    # -------------------------------------------------------------------------
    # Step 9: Save Pipeline
    # -------------------------------------------------------------------------
    print("\n💾 Step 9: Saving Pipeline")
    print("-" * 40)
    
    pipeline = {
        'feature_engineer': feature_engineer,
        'processor': processor,
        'model': model,
        'feature_columns': processor.feature_columns,
        'product_columns': model.product_columns
    }
    
    joblib.dump(pipeline, 'client_recommendation_pipeline.pkl')
    print("Pipeline saved to 'client_recommendation_pipeline.pkl'")
    
    return pipeline

# ============================================================================
# PART 5: PRODUCTION INFERENCE
# ============================================================================

def predict_for_client(client_data: Dict) -> Dict:
    """
    Make recommendations for a new client
    
    Args:
        client_data: Dict with keys: AgeGroup, Gender, MaritalStatus, IncomeBracket
        (Note: No YearsExperience needed!)
    
    Returns:
        Top product recommendations
    """
    
    # Load pipeline
    pipeline = joblib.load('client_recommendation_pipeline.pkl')
    
    # Create DataFrame
    df_client = pd.DataFrame([client_data])
    
    # Process features
    X_client = pipeline['processor'].prepare_features(df_client, is_training=False)
    
    # Get recommendations
    recommendations = pipeline['model'].recommend_top_k(X_client, k=3)
    
    return recommendations[0]

# ============================================================================
# EXECUTION
# ============================================================================

if __name__ == "__main__":
    pipeline = main()
    
    print("\n" + "=" * 80)
    print("EXAMPLE: New Client Prediction")
    print("=" * 80)
    
    # Note: No YearsExperience needed!
    new_client = {
        'AgeGroup': '30s',
        'Gender': 'Female',
        'MaritalStatus': 'Married with dependents',
        'IncomeBracket': '>50k'
    }
    
    print("\n🆕 New Client Profile:")
    for key, value in new_client.items():
        print(f"  {key:20s}: {value}")
    
    recommendations = predict_for_client(new_client)
    
    print("\n🎯 Product Recommendations:")
    for rec in recommendations['recommendations']:
        print(f"  {rec['rank']}. {rec['product']:25s} (probability: {rec['probability']:.1%}, confidence: {rec['confidence']})")
    
    print("\n✅ Complete! Recommendations based purely on CLIENT characteristics.")

CLIENT-FOCUSED PRODUCT RECOMMENDATION ENGINE
(No Adviser Experience Features)

📁 Step 1: Data Loading
----------------------------------------
Loaded 3748 samples
Filtered to 3712 clients with at least one rating

🎯 Step 2: Prepare Features and Targets
----------------------------------------
Client features: ['AgeGroup', 'Gender', 'MaritalStatus', 'IncomeBracket']
Number of products: 44
Average products per client (any interest): 10.50
Overall positive rate: 23.87%

🔧 Step 3: Client Feature Engineering
----------------------------------------
Engineered features: 35 features
Sample features: ['Age_Ordinal', 'Income_Ordinal', 'Is_Male', 'Is_Female', 'Is_Single', 'Is_Married', 'Has_Dependents', 'Is_Young', 'Is_Middle_Aged', 'Is_Senior']

✂️ Step 4: Train-Test Split
----------------------------------------
Stratification failed: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.
Using random split instead
